# Argomenti nel tempo — la topic extraction vera, seguita anno per anno

Il salto rispetto ai notebook 01 e 04: lì i temi li **proponevamo noi** (parole
chiave, famiglie-ancora) e misuravamo la somiglianza. Qui no — i temi li
**estrae il dato**: raggruppiamo i passaggi per significato, lasciamo emergere gli
argomenti, diamo a ciascun passaggio la sua **etichetta-argomento**, e poi
**contiamo** quelle etichette nel tempo. Non "quanto somiglia a X", ma "quanti
passaggi sono di argomento X, anno per anno".

Un punto di metodo che conta: un **unico modello condiviso** su tutti e quattro i
Papi (non uno per Papa), così l'argomento *k* vuol dire la stessa cosa per tutti e
i confronti **tra** Papi reggono. Poi guardiamo gli argomenti scorrere *dentro*
ogni pontificato e *tra* un pontificato e l'altro.

## Come gira

Legge i vettori già nell'indice via `ponte.tabella()` (torch-free), raggruppa con
`KMeans` e dà un nome a ogni gruppo con le sue parole più caratteristiche
(c-TF-IDF). Serve `scikit-learn`.

> ⚠️ **Ambiente.** `KMeans` usa OpenMP: su alcuni env conda Windows con MKL/torch
> può crashare (doppio `libiomp5md.dll`). Gira in un ambiente "sano" (numpy
> non-MKL, es. `conda install nomkl`, o pip-only). In sviluppo il clustering è
> stato fatto a parte e qui se ne mostra il risultato.

In [ ]:
import collections
import numpy as np
from ponte import tabella

tab = tabella()
t = tab.search().where("escludibile=false").select(
    ["papa", "data", "testo", "vector"]).limit(10**9).to_arrow()
papi = np.array(t.column("papa").to_pylist())
anno = np.array([d[:4] if d else "" for d in t.column("data").to_pylist()])
testi = t.column("testo").to_pylist()
V = np.stack(t.column("vector").to_pylist()).astype("float32")
print(len(V), "passaggi")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

K = 20
lab = KMeans(n_clusters=K, n_init=4, random_state=0).fit(V).labels_   # modello condiviso

# etichette: parole frequenti nel gruppo e rare fra i gruppi (c-TF-IDF)
STOP = ("dell della delle degli dei del nel nella nelle negli nei alla alle agli "
        "dalla dalle dello sulla sulle perche perch egli essa esso quale quali quello "
        "quella quelle questi queste questo questa cari care caro carissimi fratelli "
        "sorelle signore signori signor amici saluto saluti rivolgo ringrazio grazie "
        "cordiale particolare gioia mondo vita anni anno giorno giorni oggi sempre molto "
        "bene quindi inoltre infatti presso desidero vorrei voluto presente occasione "
        "cosi nell siete sono tutti vostra vostro hanno copyright libreria editrice "
        "vaticana bollettino stampa your vous para chiesa cristo gesu amen imparto "
        "benedica novelli benedictus niech ioannes").split()
vec = TfidfVectorizer(stop_words=STOP, token_pattern=r"(?u)\b[a-zàèéìòù]{4,}\b",
                      max_features=4000, min_df=2, sublinear_tf=True)
clus = [" ".join(testi[i] for i in range(len(V)) if lab[i] == c) for c in range(K)]
X = vec.fit_transform(clus); terms = np.array(vec.get_feature_names_out())
etich = [" ".join(terms[X[c].toarray().ravel().argsort()[::-1][:4]]) for c in range(K)]
for c in range(K):
    print(f"{int((lab==c).sum()):6}  {etich[c]}")

## La heatmap: argomento × anno

Ogni colonna è un anno; ogni riga un argomento estratto; il colore è la **quota di
passaggi** di quell'anno finiti in quell'argomento. Così si legge a occhio:
- **righe piene e continue** = argomenti che ci sono *sempre* (continuità);
- **macchie** circoscritte = argomenti **a ondate**, spesso legati a un Papa o a un
  periodo.

Gli argomenti sono ordinati per **anno di picco** (quando pesano di più): le onde
formano una specie di fiume dal passato a oggi. Le linee verticali segnano i cambi
di Papa.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

anni = sorted({y for y in anno if y})
tot = {y: int((anno == y).sum()) for y in anni}
anni = [y for y in anni if tot[y] >= 50]
M = np.array([[100 * ((lab == c) & (anno == y)).sum() / tot[y] for y in anni] for c in range(K)])

ordine = np.argsort([int(anni[M[c].argmax()]) for c in range(K)])
M, etich = M[ordine], [etich[i] for i in ordine]

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(M, ax=ax, cmap="rocket_r", yticklabels=etich, xticklabels=anni,
            cbar_kws={"label": "% dei passaggi dell'anno"})
for i, y in enumerate(anni):
    if y in ("2005", "2013", "2025"):
        ax.axvline(i, color="cyan", lw=1.2)
ax.set_xlabel("anno  (linee: cambio di Papa — 2005 Benedetto, 2013 Francesco, 2025 Leone)")
ax.set_title("Gli argomenti ESTRATTI dai testi, seguiti nel tempo\n"
             "righe piatte = continui · macchie = a ondate", fontsize=12)
plt.setp(ax.get_xticklabels(), fontsize=7, rotation=90)
plt.setp(ax.get_yticklabels(), fontsize=8, rotation=0)
fig.tight_layout(); fig.savefig("argomenti-nel-tempo.png", dpi=150, bbox_inches="tight")
print("salvata argomenti-nel-tempo.png")

![argomenti estratti nel tempo](argomenti-nel-tempo.png)

## Come si legge

La figura risponde proprio alla domanda *continuità o ondate?* — e la risposta è
**tutte e due, su piani diversi**:

- **Continuità.** Alcuni argomenti sono righe quasi piene per tutto l'arco: la
  predicazione del Vangelo, la famiglia/gli sposi, il mariano, l'ecumenismo. Sono
  il fondo che non cambia da un Papa all'altro.
- **Onde legate al Papa.** Altri sono macchie nette: le visite *ad limina* e
  l'America Latina pesano nell'era **Giovanni Paolo II** e poi si abbassano;
  la **Santa Marta** (le omelie quotidiane) è un blocco tutto di **Francesco**
  (2013-2018); il filone **sociale/attualità** (disarmo, agricoltura, commercio,
  migranti) si accende nell'era **Francesco-Leone**.
- **Onde brevi.** Alcune macchie sono eventi: un anniversario, un Giubileo, un
  viaggio — argomenti che salgono un anno e spariscono.

Quindi: il filo resta (continuità del fondo), ma gli **accenti scorrono** — e
scorrono in buona parte ai **cambi di Papa**, non a caso lungo il tempo.

---
*Nota. Gli argomenti sono **estratti dal dato** (raggruppamento per significato),
non temi imposti da noi; le etichette sono le loro **parole-chiave automatiche** —
indicative, qualcuna grezza. Solo viste aggregate (quote per anno), nessun testo
riprodotto (© Libreria Editrice Vaticana).*